# Duka Electronics Sales Exploration

## Executive Summary

- Across **1,000 orders**, the dataset covers January–June 2026.
- **Nairobi CBD** led branch revenue with **KES 10,512,000** from **284 orders**.
- **Laptops** led revenue at **KES 11,520,000**, while **smartphones** led volume with **313 units**.
- **May** was the best month at **KES 7,458,000** and **February** the weakest at **KES 5,852,000**, a **27.4%** increase from the low to the high.
- Weekend orders averaged **KES 41,702** across **302 orders**, versus **KES 40,567** on weekdays; weekday orders still generated more total revenue (**KES 28,316,000** versus **KES 12,594,000**).

## First-five-minutes inspection

Load the generated file with `order_date` parsed as a date, then check its
shape, data types, missingness, date coverage and key category values before
calculating any business metrics. This initial scan confirms whether the file
looks like the intended 1,000-order half-year extract and identifies obvious
problems early.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

if not Path("duka_sales.csv").is_file():
    from generate_data import generate_duka_sales

    generate_duka_sales().to_csv("duka_sales.csv", index=False, date_format="%Y-%m-%d")

sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
print("Shape:", sales.shape)
display(sales.dtypes.rename("dtype").to_frame())
display(sales.isna().sum().rename("missing_rows").to_frame())
print("Date range:", sales["order_date"].min().date(), "to", sales["order_date"].max().date())
for column in ["branch", "product", "channel"]:
    print(f"{column} categories:", sorted(sales[column].dropna().unique().tolist()))
display(sales.head())

Shape: (1000, 7)


,dtype
order_id,str
order_date,datetime64[us]
branch,str
product,str
units,int64
unit_price,int64
channel,str


,missing_rows
order_id,0
order_date,0
branch,0
product,0
units,0
unit_price,0
channel,0


Date range: 2026-01-01 to 2026-06-30
branch categories: ['Kisumu', 'Mombasa', 'Nairobi CBD', 'Nairobi West', 'Nakuru']
product categories: ['Headphones', 'Laptop', 'Monitor', 'Printer', 'Smartphone', 'Tablet']
channel categories: ['In-store', 'Online']


,order_id,order_date,branch,product,units,unit_price,channel
0,DU00001,2026-04-02,Nairobi CBD,Smartphone,1,35000,Online
1,DU00002,2026-05-26,Mombasa,Monitor,1,24000,In-store
2,DU00003,2026-02-15,Nairobi West,Tablet,1,28000,In-store
3,DU00004,2026-01-18,Kisumu,Tablet,1,28000,In-store
4,DU00005,2026-05-03,Kisumu,Smartphone,1,35000,In-store


## Create derived columns

In [2]:
sales["revenue"] = sales["units"] * sales["unit_price"]
sales["month"] = sales["order_date"].dt.month_name()
sales["weekday"] = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5
sales["price_band"] = np.where(sales["unit_price"] >= 28_000, "Premium", "Value")
display(sales.head())

,order_id,order_date,branch,product,units,unit_price,channel,revenue,month,weekday,is_weekend,price_band
0,DU00001,2026-04-02,Nairobi CBD,Smartphone,1,35000,Online,35000,April,Thursday,False,Premium
1,DU00002,2026-05-26,Mombasa,Monitor,1,24000,In-store,24000,May,Tuesday,False,Value
2,DU00003,2026-02-15,Nairobi West,Tablet,1,28000,In-store,28000,February,Sunday,True,Premium
3,DU00004,2026-01-18,Kisumu,Tablet,1,28000,In-store,28000,January,Sunday,True,Premium
4,DU00005,2026-05-03,Kisumu,Smartphone,1,35000,In-store,35000,May,Sunday,True,Premium


## Task 3 — Branch revenue and order rankings

In [3]:
branch_summary = (
    sales.groupby("branch")
    .agg(total_revenue=("revenue", "sum"), orders=("order_id", "count"))
    .sort_values("total_revenue", ascending=False)
)
branch_summary["revenue_rank"] = branch_summary["total_revenue"].rank(method="min", ascending=False).astype(int)
branch_summary["order_rank"] = branch_summary["orders"].rank(method="min", ascending=False).astype(int)
display(branch_summary)

,total_revenue,orders,revenue_rank,order_rank
branch,,,,
Nairobi CBD,10512000,284,1,1
Nairobi West,9155000,218,2,2
Mombasa,8316000,180,3,3
Kisumu,7256000,176,4,4
Nakuru,5671000,142,5,5


The rankings are the same: Nairobi CBD, Nairobi West, Mombasa, Kisumu,
then Nakuru. Order volume is the main driver in this sample; differences in
product and unit mix do not change the ordering by total revenue.

## Task 4 — Top products by revenue and units

In [4]:
product_summary = sales.groupby("product").agg(
    total_revenue=("revenue", "sum"), units_sold=("units", "sum")
)
print("Top three by revenue")
display(product_summary.sort_values("total_revenue", ascending=False).head(3))
print("Top three by units sold")
display(product_summary.sort_values("units_sold", ascending=False).head(3))

Top three by revenue


,total_revenue,units_sold
product,,
Laptop,11520000,160
Smartphone,10955000,313
Tablet,6440000,230


Top three by units sold


,total_revenue,units_sold
product,,
Smartphone,10955000,313
Headphones,1960000,245
Monitor,5760000,240


The lists differ: laptops, smartphones and tablets lead revenue, while
smartphones, headphones and monitors lead units. Laptops contribute the most
revenue despite fewer units because their KES 72,000 unit price is much higher;
headphones and monitors sell more units at lower prices.

## Task 5 — Weekend versus weekday orders

In [5]:
weekend_summary = sales.groupby("is_weekend").agg(
    orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    average_order_revenue=("revenue", "mean"),
)
weekend_summary.index = weekend_summary.index.map({False: "Weekday", True: "Weekend"})
display(weekend_summary.round(2))

,orders,total_revenue,average_order_revenue
is_weekend,,,
Weekday,698,28316000,40567.34
Weekend,302,12594000,41701.99


Weekends average KES 41,702 per order, about 2.8% above the weekday
average of KES 40,567. Weekdays nevertheless contribute much more total
revenue—KES 28,316,000 versus KES 12,594,000—because there are 698 weekday
orders compared with 302 weekend orders.

## Task 6 — Best and worst month

In [6]:
monthly_revenue = sales.groupby("month")["revenue"].sum().reindex(
    ["January", "February", "March", "April", "May", "June"]
)
best_month = monthly_revenue.idxmax()
worst_month = monthly_revenue.idxmin()
percentage_difference = (monthly_revenue.max() / monthly_revenue.min() - 1) * 100
display(monthly_revenue.rename("revenue").to_frame())
print(f"Best: {best_month}; worst: {worst_month}; difference: {percentage_difference:.1f}%")

,revenue
month,
January,7222000
February,5852000
March,6325000
April,7311000
May,7458000
June,6742000


Best: May; worst: February; difference: 27.4%


May is the best month at KES 7,458,000, and February is the worst at
KES 5,852,000. May revenue is 27.4% higher than February, using the worst
month as the comparison base.

## Task 7 — Premium online orders in Nairobi and Mombasa

In [7]:
premium_online_coast_capital = sales.loc[
    sales["price_band"].eq("Premium")
    & sales["channel"].eq("Online")
    & sales["branch"].isin(["Nairobi CBD", "Nairobi West", "Mombasa"])
].copy()
Path("outputs").mkdir(exist_ok=True)
premium_online_coast_capital.to_csv(
    "outputs/premium_online_coast_capital.csv", index=False, date_format="%Y-%m-%d"
)
print("Matching orders:", len(premium_online_coast_capital))
print("Total revenue (KES):", int(premium_online_coast_capital["revenue"].sum()))
display(premium_online_coast_capital.head())

Matching orders: 104
Total revenue (KES): 6264000


,order_id,order_date,branch,product,units,unit_price,channel,revenue,month,weekday,is_weekend,price_band
0,DU00001,2026-04-02,Nairobi CBD,Smartphone,1,35000,Online,35000,April,Thursday,False,Premium
7,DU00008,2026-04-15,Nairobi CBD,Smartphone,1,35000,Online,35000,April,Wednesday,False,Premium
23,DU00024,2026-02-25,Mombasa,Laptop,1,72000,Online,72000,February,Wednesday,False,Premium
60,DU00061,2026-03-14,Nairobi CBD,Tablet,2,28000,Online,56000,March,Saturday,True,Premium
61,DU00062,2026-02-18,Mombasa,Smartphone,1,35000,Online,35000,February,Wednesday,False,Premium


There are **104** qualifying orders, totaling **KES 6,264,000**.
The exported file contains only Premium Online orders from the two Nairobi
branches and Mombasa; it is written without a DataFrame index.

## Task 8 — Excel branch summary and revenue PivotTable

In [8]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill

Path("excel").mkdir(exist_ok=True)
workbook = Workbook()
data_sheet = workbook.active
data_sheet.title = "Sales Data"
excel_sales = sales[["order_id", "order_date", "branch", "product", "units", "unit_price", "channel", "revenue"]]
data_sheet.append(excel_sales.columns.tolist())
for row in excel_sales.itertuples(index=False, name=None):
    data_sheet.append(list(row))

summary_sheet = workbook.create_sheet("Branch Summary")
summary_sheet.append(["Branch", "Revenue (KES)", "Region", "Orders"])
branch_order = branch_summary.index.tolist()
regions = {
    "Nairobi CBD": "Nairobi",
    "Nairobi West": "Nairobi",
    "Mombasa": "Coast",
    "Kisumu": "Western",
    "Nakuru": "Rift Valley",
}
for row_number, branch in enumerate(branch_order, start=2):
    summary_sheet.cell(row_number, 1, branch)
    summary_sheet.cell(row_number, 2, f"=SUMIFS('Sales Data'!$H:$H,'Sales Data'!$C:$C,A{row_number})")
    summary_sheet.cell(row_number, 3, f'=XLOOKUP(A{row_number},$E$2:$E$6,$F$2:$F$6,"Unknown")')
    summary_sheet.cell(row_number, 4, f'=COUNTIF(\'Sales Data\'!$C:$C,A{row_number})')
summary_sheet["E1"] = "Branch lookup"
summary_sheet["F1"] = "Region"
for row_number, (branch, region) in enumerate(regions.items(), start=2):
    summary_sheet.cell(row_number, 5, branch)
    summary_sheet.cell(row_number, 6, region)

pivot_sheet = workbook.create_sheet("Revenue Pivot")
pivot = sales.pivot_table(
    index="branch", columns="channel", values="revenue", aggfunc="sum", fill_value=0
)
channels = [channel for channel in ["In-store", "Online"] if channel in pivot.columns]
pivot_sheet.append(["PivotTable — Revenue by branch and channel", *channels, "Grand Total"])
for branch, values in pivot.iterrows():
    row_values = [branch, *[int(values[channel]) for channel in channels], int(values.sum())]
    pivot_sheet.append(row_values)
total_row = pivot_sheet.max_row + 1
pivot_sheet.cell(total_row, 1, "Grand Total")
for column_number, channel in enumerate(channels, start=2):
    pivot_sheet.cell(total_row, column_number, int(pivot[channel].sum()))
pivot_sheet.cell(total_row, len(channels) + 2, int(pivot.to_numpy().sum()))

for sheet in workbook.worksheets:
    sheet.freeze_panes = "A2"
    for cell in sheet[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="1F4E78")
    for column_cells in sheet.columns:
        width = min(max(len(str(cell.value or "")) for cell in column_cells) + 2, 34)
        sheet.column_dimensions[column_cells[0].column_letter].width = width
workbook.calculation.fullCalcOnLoad = True
workbook.calculation.forceFullCalc = True
workbook.calculation.calcMode = "auto"
workbook.save("excel/duka_branch_summary.xlsx")
print("Pandas branch revenue total:", int(branch_summary["total_revenue"].sum()))
print("PivotTable grand total:", int(pivot.to_numpy().sum()))
print("Excel includes SUMIFS revenue formulas and XLOOKUP region formulas.")

Pandas branch revenue total: 40910000
PivotTable grand total: 40910000
Excel includes SUMIFS revenue formulas and XLOOKUP region formulas.


The workbook includes the source rows, a revenue-ranked branch table
with `SUMIFS` and `XLOOKUP` formulas, and a branch-by-channel PivotTable
summary. The PivotTable grand total and pandas branch-revenue total are both
KES 40,910,000, so the spreadsheet aggregation reconciles to the notebook.